# 12장 — 🎯 캡스톤: 사내 AI 도입, 모형으로 미리 돌려 보기

웹 챕터(12장)와 **같은 모형, 같은 수치**를 파이썬으로 계산합니다:
Bass 확산 모형(+ 이탈)을 저량-유량으로 돌려 일곱 시나리오를 비교하고, 이론값(정점 시각·정점 속도·평형)과 맞춰 봅니다.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch12')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 모형: 잠재 사용자 → 사용자 (→ 다시 잠재 사용자)

$$\text{도입} = \Big(p + q\,\frac{A}{M}\Big)(M - A), \qquad \text{이탈} = \frac{A}{L}, \qquad \frac{dA}{dt} = \text{도입} - \text{이탈}$$

직원 M = 3,000명, 처음 사용자 0명, 36개월. 오일러법을 DT = 1/64개월로 잘게 썰어 이론값과 소수 첫째 자리까지 맞춥니다(웹 sim.ts adoptionRun 과 같은 계산).
L = 0 은 '이탈 없음'으로 씁니다.

In [ ]:
import math

M, T, DT = 3000, 36, 1 / 64

def adoption(p, q, L=0):
    P, A, prevA = M, 0.0, 0.0
    t50, peak_t, peak = None, 0.0, p * M
    path = [(0.0, 0.0, p * M)]
    for i in range(1, round(T / DT) + 1):
        a = (p + q * A / M) * P
        c = A / L if L > 0 else 0.0
        P, A = P + (c - a) * DT, A + (a - c) * DT
        now = i * DT
        rate = (p + q * A / M) * P
        if rate > peak + 1e-12:
            peak, peak_t = rate, now
        if t50 is None and prevA < M / 2 <= A:
            t50 = now - DT + (M / 2 - prevA) / (A - prevA) * DT
        prevA = A
        path.append((now, A, rate))
    eq = 1.0
    if L > 0 and q > 0:
        b = q - p - 1 / L
        eq = (b + math.sqrt(b * b + 4 * q * p)) / (2 * q)
    return dict(t50=t50, peak_t=peak_t, peak=peak, A36=A, eq=eq, path=path)

scenarios = {
    '기본': (0.01, 0.4, 0),
    '홍보 집중': (0.03, 0.4, 0),
    '입소문 집중': (0.01, 0.6, 0),
    '현실: 이탈 있음(10개월)': (0.01, 0.4, 10),
    '이탈 방지(20개월)': (0.01, 0.4, 20),
    '홍보 집중 + 이탈': (0.03, 0.4, 10),
    '입소문 집중 + 이탈': (0.01, 0.6, 10),
}
res = {name: adoption(*args) for name, args in scenarios.items()}
print(f'{"시나리오":18s} 절반 도달  도입 정점(개월, 명/월)  36개월 사용자  평형')
for name, r in res.items():
    print(f'{name:18s} {r["t50"]:6.2f}   {r["peak_t"]:6.2f}, {r["peak"]:6.1f}      {r["A36"]:8.1f}   {r["eq"] * 100:6.2f}%')

In [ ]:
# 웹 실험실(AdoptionLab)·본문 수치 자가검증
b = res['기본']
assert round(b['t50'], 1) == 9.1 and round(b['peak_t'], 1) == 9.0 and round(b['peak']) == 315 and round(b['A36']) == 3000
assert round(res['홍보 집중']['t50'], 1) == 6.4 and round(b['t50'] - res['홍보 집중']['t50'], 1) == 2.8
assert round(res['입소문 집중']['t50'], 1) == 6.8 and round(res['입소문 집중']['peak']) == 465
r10 = res['현실: 이탈 있음(10개월)']
assert round(r10['eq'] * 100, 1) == 75.8 and round(r10['eq'] * M) == 2274 and round(r10['t50'], 1) == 12.2
assert round(res['이탈 방지(20개월)']['eq'] * 100, 1) == 87.8
assert round(res['홍보 집중 + 이탈']['eq'] * 100, 1) == 77.2
assert round(res['입소문 집중 + 이탈']['eq'] * 100, 1) == 83.7 and round(res['입소문 집중 + 이탈']['t50'], 1) == 8.1
print('✅ 웹과 일치: 기본 절반 9.1개월·정점 9.0개월(월 315명) · 홍보는 2.8개월 당김 · 이탈 10개월이면 평형 75.8%')

## 2. 이론값과 맞춰 보기

이탈이 없는 Bass 모형에는 정확한 답이 있습니다.

$$F(t) = \frac{1 - e^{-(p+q)t}}{1 + \frac{q}{p} e^{-(p+q)t}}, \qquad t^* = \frac{\ln(q/p)}{p + q}, \qquad \text{정점 속도} = \frac{M (p+q)^2}{4q}$$

이탈이 있으면 평형은 $(p + qa)(1 - a) = a/L$ 의 근이고, p가 작을 때 대략 $1 - 1/(qL)$.

In [ ]:
p, q = 0.01, 0.4
F = lambda t: (1 - math.exp(-(p + q) * t)) / (1 + q / p * math.exp(-(p + q) * t))
t_star = math.log(q / p) / (p + q)
peak_theory = M * (p + q) ** 2 / (4 * q)
print(f'이론: 정점 {t_star:.3f}개월, 정점 속도 {peak_theory:.2f}명/월, 정점 때 사용자 비율 {F(t_star):.4f} (= 1/2 − p/(2q) = {0.5 - p / (2 * q):.4f})')
assert round(t_star, 1) == 9.0 and round(peak_theory, 1) == 315.2
assert abs(F(t_star) - (0.5 - p / (2 * q))) < 1e-12
# 오일러(DT = 1/64)와 정확한 답의 차이
err = max(abs(A - M * F(t)) for t, A, _ in res['기본']['path'])
print(f'오일러(DT = 1/64)와 정확한 답의 최대 차이: {err:.2f}명')
assert err < 10          # 직원 3,000명의 0.3% 이내
# 계산 연습: ln 40 ÷ 0.41, 3000 × 0.41² ÷ 1.6
assert abs(math.log(40) / 0.41 - 9.0) < 0.01 and abs(3000 * 0.41 ** 2 / 1.6 - 315.2) < 0.05
# 평형 근사 1 − 1/(qL)
for qq in [0.4, 0.6]:
    exact_eq = adoption(0.01, qq, 10)['eq']
    print(f'q = {qq}: 평형 {exact_eq * 100:.1f}%  vs  근사 1 − 1/(qL) = {(1 - 1 / (qq * 10)) * 100:.1f}%')
print('✅ 이론값과 일치: t* = 9.0개월, 정점 315.2명/월, 정점은 사용자 비율 48.75%(절반 바로 앞)')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
colors = ['#4f46e5', '#0ea5e9', '#059669', '#e11d48', '#f59e0b', '#a855f7', '#14b8a6']
for (name, r), c in zip(res.items(), colors):
    ts = [x[0] for x in r['path']][::16]
    ax1.plot(ts, [x[1] for x in r['path']][::16], color=c, lw=2, label=name)
    ax2.plot(ts, [x[2] for x in r['path']][::16], color=c, lw=1.8, label=name)
ax1.axhline(M, ls='--', color='#2563eb', lw=1); ax1.axhline(M / 2, ls=':', color='#d97706', lw=1)
ax1.set_xlabel('개월'); ax1.set_ylabel('사용자 수(명)'); ax1.set_title('시나리오별 사용자 수'); ax1.legend(fontsize=7); ax1.grid(alpha=0.3)
ax2.set_xlabel('개월'); ax2.set_ylabel('도입(명/월)'); ax2.set_title('시나리오별 도입 속도'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 도입 전략 판정서 (견본)

- **홍보·교육(p)은 시작을 당긴다**: 절반 도달 9.1 → 6.4개월. 이탈이 있을 때 평형은 거의 그대로(75.8% → 77.2%).
- **입소문(q)은 속도와 수준을 함께 올린다**: 절반 6.8개월, 이탈이 있을 때 평형 83.7%.
- **이탈 방지(L)는 최종 수준을 정한다**: 사용 기간 10 → 20개월이면 평형 75.8% → 87.8%.
- **신규 가입이 꺾이면 절반쯤 왔다는 신호**(정점 = 사용자 비율 1/2 − p/(2q)). 이때부터 이탈 지표를 본다.
- 실제 p·q 는 초기 숫자로 추정하되 오차범위를 붙이고(10부 6장) 계속 갱신한다. 구조적 결론을 먼저 쓴다.

**🎓 12부 완주!** 웹 챕터로 돌아가 마지막 연습 10문제를 풀면 졸업입니다.